# 主成分风险平价模型复现研究
## Decay-Weighted PCA Risk Parity Model

本Notebook复现天风证券研报《资产配置策略研究之二：引入衰减加权和趋势跟踪的主成分风险平价模型研究》

**研报核心内容：**
1. 传统风险平价模型的局限性分析
2. 主成分风险平价(PCRP)模型
3. 衰减加权法用于预期风险估计
4. 趋势跟踪法用于预期走势估计
5. WDC-PCRP模型实证分析

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from source.data_loader import DataLoader, convert_market_code
from source.config import (
    DEFAULT_START_DATE,
    DEFAULT_END_DATE,
    DECAY_WEIGHTING_PARAMS,
    TF_PARAMS
)
from source.decay_weighting import DecayWeighting, create_decay_weighting
from source.trend_following import TrendFollowing, create_trend_following
from source.pc_risk_parity import (
    PrincipalComponentsRiskParity,
    PCRPwithTrendFollowing,
    WDCPCRP,
    create_pcrp_model
)
from source.backtest import BacktestEngine, MultiStrategyBacktest

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('模块导入成功')

## 1. 数据获取

In [ ]:
# 初始化数据加载器
loader = DataLoader()

print('数据加载器初始化完成')
print(f'Tushare API URL: {loader.api_url}')

In [ ]:
# 定义资产配置
# 根据研报，我们使用以下资产类别

asset_configs = {
    '000300.SH': {'name': '沪深300', 'type': 'index'},
    '000016.SH': {'name': '上证50', 'type': 'index'},
    '000905.SH': {'name': '中证500', 'type': 'index'},
    '000012.SH': {'name': '上证国债', 'type': 'bond'},
    '000013.SH': {'name': '上证企债', 'type': 'bond'},
}

assets = list(asset_configs.keys())
asset_names = [config['name'] for config in asset_configs.values()]

In [ ]:
# 获取真实市场数据
import tushare as ts

# 初始化tushare API
token = '1015d5b62774ab54f44bc6ef3ed95b02e2d8fb9a68dac06c0e344a3f3ceb'
pro = ts.pro_api(token)
pro._DataApi__token = token
pro._DataApi__http_url = 'http://jiaoch.site'

# 定义要获取的资产
index_codes = ['000300.SH', '000016.SH', '000905.SH']
bond_codes = ['000012.SH', '000013.SH']

# 获取指数数据
all_prices = {}

print('正在获取市场数据...')

# 获取股票指数数据
for code in index_codes:
    try:
        df = pro.index_daily(ts_code=code, start_date='20100101', end_date='20171117')
        df['trade_date'] = pd.to_datetime(df['trade_date'])
        df = df.sort_values('trade_date')
        df = df.set_index('trade_date')
        name = {'000300.SH': '沪深300', '000016.SH': '上证50', '000905.SH': '中证500'}[code]
        all_prices[name] = df['close']
        print(f'  {name}: {len(df)} 条记录')
    except Exception as e:
        print(f'  获取 {code} 失败: {e}')

# 获取债券指数数据
for code in bond_codes:
    try:
        # 注意：债券指数数据可能需要通过其他接口获取
        df = pro.bond_daily(ts_code=code, start_date='20100101', end_date='20171117')
        if df is not None and not df.empty:
            df['trade_date'] = pd.to_datetime(df['trade_date'])
            df = df.sort_values('trade_date')
            df = df.set_index('trade_date')
            name = {'000012.SH': '上证国债', '000013.SH': '上证企债'}[code]
            all_prices[name] = df['close']
            print(f'  {name}: {len(df)} 条记录')
        else:
            print(f'  {code} 无债券数据')
    except Exception as e:
        print(f'  获取 {code} 失败: {e}')

# 创建价格DataFrame
prices = pd.DataFrame(all_prices)
prices = prices.sort_index()
prices = prices.dropna()

# 计算收益率
returns = prices.pct_change().dropna()

asset_names = list(prices.columns)
print(f'\n数据时间范围: {prices.index[0].strftime("%Y-%m-%d")} 至 {prices.index[-1].strftime("%Y-%m-%d")}')
print(f'数据点数量: {len(prices)}')
print(f'资产数量: {len(asset_names)}')
print(f'资产列表: {asset_names}')

In [ ]:
# 绘制资产价格走势图
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 价格走势图
prices.plot(ax=axes[0], linewidth=1.5)
axes[0].set_title('资产价格走势', fontsize=14)
axes[0].set_xlabel('日期')
axes[0].set_ylabel('价格')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# 收益分布
returns.plot(kind='hist', bins=50, ax=axes[1], alpha=0.7, density=True)
axes[1].set_title('资产收益分布', fontsize=14)
axes[1].set_xlabel('日收益率')
axes[1].set_ylabel('密度')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/asset_prices_returns.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. 资产特性分析

根据研报，资产的特性分析包括：
1. 资产收益率的难预测性
2. 资产波动率的聚集性和衰减性
3. 资产间相关系数的长期关联性

In [ ]:
# 计算资产统计特性
asset_stats = pd.DataFrame({
    '年化收益率': returns.mean() * 252,
    '年化波动率': returns.std() * np.sqrt(252),
    '夏普比率': (returns.mean() * 252) / (returns.std() * np.sqrt(252)),
    '最大回撤': returns.apply(lambda x: (x.cumsum() - x.cumsum().cummax()).min())
})

print('资产统计特性：')
print(asset_stats.round(4))

In [ ]:
# 计算相关系数矩阵
correlation_matrix = returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdYlBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('资产收益相关系数矩阵', fontsize=14)
plt.tight_layout()
plt.savefig('../output/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 衰减加权法

实现衰减加权法用于预期风险估计，给近期数据更高的权重。

In [ ]:
# 创建衰减加权估计器
decay_weighting = DecayWeighting(
    returns,
    volatility_half_life=DECAY_WEIGHTING_PARAMS['volatility_half_life'],
    correlation_half_life=DECAY_WEIGHTING_PARAMS['correlation_half_life']
)

# 计算衰减加权波动率
decay_volatility = decay_weighting.calculate_decay_weighted_volatility()

# 计算标准波动率
std_volatility = returns.std() * np.sqrt(252)

volatility_comparison = pd.DataFrame({
    '标准波动率': std_volatility,
    '衰减加权波动率': decay_volatility,
    '差异': decay_volatility - std_volatility
})

print('波动率对比（年化）：')
print(volatility_comparison.round(4))

In [ ]:
# 计算衰减加权相关矩阵
decay_correlation = decay_weighting.calculate_decay_weighted_correlation()
decay_correlation_df = pd.DataFrame(
    decay_correlation,
    index=asset_names,
    columns=asset_names
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 标准相关矩阵
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.3f',
    cmap='RdYlBu_r',
    center=0,
    square=True,
    ax=axes[0]
)
axes[0].set_title('标准相关矩阵', fontsize=12)

# 衰减加权相关矩阵
sns.heatmap(
    decay_correlation_df,
    annot=True,
    fmt='.3f',
    cmap='RdYlBu_r',
    center=0,
    square=True,
    ax=axes[1]
)
axes[1].set_title('衰减加权相关矩阵', fontsize=12)

plt.tight_layout()
plt.savefig('../output/decay_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. 趋势跟踪法

实现趋势跟踪法用于预期走势估计，基于移动平均线交叉判断市场趋势。

In [ ]:
# 创建趋势跟踪估计器
trend_following = TrendFollowing(
    prices,
    short_ma=TF_PARAMS['short_ma'],
    long_ma=TF_PARAMS['long_ma'],
    lookback_period=TF_PARAMS['lookback_period']
)

# 获取趋势信号
trend_signals = trend_following.calculate_ma_crossover_signal()

# 计算趋势强度
trend_strength = trend_following.calculate_trend_strength()

print('最近10个交易日的趋势信号：')
print(trend_signals.tail(10))

In [ ]:
# 可视化趋势信号和价格
asset_idx = 0  # 选择第一个资产
asset = asset_names[asset_idx]

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 价格和移动平均线
short_ma, long_ma = trend_following.calculate_moving_averages()

axes[0].plot(prices.index, prices[asset], label='价格', alpha=0.7)
axes[0].plot(prices.index, short_ma[asset], label=f'SMA {TF_PARAMS["short_ma"]}', linestyle='--')
axes[0].plot(prices.index, long_ma[asset], label=f'SMA {TF_PARAMS["long_ma"]}', linestyle='--')
axes[0].set_title(f'{asset} - 价格与移动平均线', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 趋势信号
axes[1].plot(trend_signals.index, trend_signals[asset], drawstyle='steps-post')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_title(f'{asset} - MA交叉信号', fontsize=12)
axes[1].set_ylabel('信号 (1=多头, -1=空头)')
axes[1].grid(True, alpha=0.3)

# 趋势强度
axes[2].plot(trend_strength.index, trend_strength[asset])
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].set_title(f'{asset} - 趋势强度', fontsize=12)
axes[2].set_ylabel('趋势强度')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/trend_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 主成分风险平价模型 (PCRP)

实现PCRP模型：
1. 使用PCA将资产变换到正交空间
2. 在主成分空间应用风险平价
3. 反推得到原始资产权重

In [ ]:
# 创建PCRP模型
pcrp = PrincipalComponentsRiskParity(
    returns,
    use_decay_weighting=False
)

# 计算权重
pcrp_weights = pcrp.calculate_weights()

# 显示权重
weights_df = pd.DataFrame({
    '资产': asset_names,
    'PCRP权重': pcrp_weights
})
weights_df = weights_df.sort_values('PCRP权重', ascending=False)

print('PCRP模型资产权重：')
print(weights_df.to_string(index=False))

In [ ]:
# 可视化权重
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(asset_names)))

bars = ax.barh(weights_df['资产'], weights_df['PCRP权重'], color=colors)

ax.set_xlabel('权重')
ax.set_title('PCRP模型资产权重分配', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')

# 添加数值标签
for bar, weight in zip(bars, weights_df['PCRP权重']):
    ax.text(weight + 0.01, bar.get_y() + bar.get_height()/2,
            f'{weight:.3f}', va='center')

plt.tight_layout()
plt.savefig('../output/pcrp_weights.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. WDC-PCRP模型（衰减加权+趋势跟踪）

结合衰减加权法和趋势跟踪法的完整WDC-PCRP模型。

In [ ]:
# 创建WDC-PCRP模型
wdc_pcrp = WDCPCRP(
    returns,
    prices,
    decay_params=DECAY_WEIGHTING_PARAMS,
    trend_params=TF_PARAMS
)

# 计算权重
wdc_weights = wdc_pcrp.calculate_weights()

# 获取组合指标
metrics = wdc_pcrp.get_portfolio_metrics()

print('WDC-PCRP模型资产权重：')
for name, weight in zip(asset_names, wdc_weights):
    print(f'  {name}: {weight:.4f}')

## 7. 回测分析

对各种模型进行回测，比较性能表现。

In [ ]:
# 定义权重生成函数

def pcrp_weights_generator(returns_df, prices_df):
    """PCRP模型权重生成器"""
    model = PrincipalComponentsRiskParity(returns_df, use_decay_weighting=False)
    return model.calculate_weights()


def pcrp_decay_weights_generator(returns_df, prices_df):
    """带衰减加权的PCRP模型权重生成器"""
    model = PrincipalComponentsRiskParity(
        returns_df,
        use_decay_weighting=True,
        decay_params=DECAY_WEIGHTING_PARAMS
    )
    return model.calculate_weights()


def wdc_pcrp_weights_generator(returns_df, prices_df):
    """WDC-PCRP模型权重生成器"""
    model = WDCPCRP(
        returns_df,
        prices_df,
        decay_params=DECAY_WEIGHTING_PARAMS,
        trend_params=TF_PARAMS
    )
    return model.calculate_weights()


def equal_weight_generator(returns_df, prices_df):
    """等权重模型权重生成器"""
    n = returns_df.shape[1]
    return np.ones(n) / n

In [ ]:
# 创建回测引擎字典
strategies = {
    'Equal Weight': equal_weight_generator,
    'Standard PCRP': pcrp_weights_generator,
    'PCRP + Decay': pcrp_decay_weights_generator,
    'WDC-PCRP': wdc_pcrp_weights_generator
}

# 分割训练集和测试集
train_end = int(len(returns) * 0.7)
train_returns = returns.iloc[:train_end]
test_returns = returns.iloc[train_end:]
train_prices = prices.iloc[:train_end]
test_prices = prices.iloc[train_end:]

print(f'训练集: {train_returns.index[0].strftime("%Y-%m-%d")} 至 {train_returns.index[-1].strftime("%Y-%m-%d")}')
print(f'测试集: {test_returns.index[0].strftime("%Y-%m-%d")} 至 {test_returns.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
# 运行所有策略回测
backtest_results = {}

for name, weight_func in strategies.items():
    print(f'运行 {name} 回测...')
    
    engine = BacktestEngine(
        test_returns,
        test_prices,
        strategy_name=name,
        initial_capital=1_000_000,
        rebalance_freq='M',
        transaction_cost=0.001
    )
    
    engine.set_weights_generator(weight_func)
    metrics = engine.run_backtest(lookback_period=60)
    
    backtest_results[name] = {
        'engine': engine,
        'metrics': metrics
    }
    
    print(f'  总收益: {metrics["total_return"]:.2%}')
    print(f'  年化收益: {metrics["annualized_return"]:.2%}')
    print(f'  夏普比率: {metrics["sharpe_ratio"]:.4f}')
    print(f'  最大回撤: {metrics["max_drawdown"]:.2%}')
    print()

In [ ]:
# 汇总回测结果
results_summary = pd.DataFrame({
    name: res['metrics']
    for name, res in backtest_results.items()
}).T

display_cols = [
    'total_return',
    'annualized_return',
    'annualized_volatility',
    'sharpe_ratio',
    'max_drawdown',
    'calmar_ratio',
    'win_rate',
    'final_value'
]

results_summary_display = results_summary[display_cols].copy()
results_summary_display.columns = [
    '总收益', '年化收益', '年化波动率', '夏普比率',
    '最大回撤', 'Calmar比率', '胜率', '最终净值'
]

print('=' * 80)
print('策略回测结果汇总')  
print('=' * 80)
print(results_summary_display.round(4).to_string())

In [ ]:
# 可视化策略比较
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 累积收益曲线
ax1 = axes[0, 0]
for name, res in backtest_results.items():
    equity = res['engine'].get_equity_curve()
    normalized_equity = equity / equity.iloc[0]
    ax1.plot(normalized_equity.index, normalized_equity, label=name, linewidth=1.5)
ax1.set_title('策略累积收益对比', fontsize=12)
ax1.set_xlabel('日期')
ax1.set_ylabel('标准化净值')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 年化收益对比
ax2 = axes[0, 1]
annualized_returns = results_summary['annualized_return'].sort_values(ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0, 1, len(annualized_returns)))
bars = ax2.barh(annualized_returns.index, annualized_returns * 100, color=colors)
ax2.set_xlabel('年化收益率 (%)')
ax2.set_title('策略年化收益对比', fontsize=12)
ax2.grid(True, alpha=0.3, axis='x')

# 3. 风险收益比对比 (夏普比率 vs Calmar比率)
ax3 = axes[1, 0]
sharpe = results_summary['sharpe_ratio']
calmar = results_summary['calmar_ratio']
colors = plt.cm.Set2(np.linspace(0, 1, len(sharpe)))
scatter = ax3.scatter(sharpe, calmar, c=range(len(sharpe)), cmap='Set2', s=200)
for i, name in enumerate(sharpe.index):
    ax3.annotate(name, (sharpe.iloc[i], calmar.iloc[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)
ax3.set_xlabel('夏普比率')
ax3.set_ylabel('Calmar比率')
ax3.set_title('风险收益比对比', fontsize=12)
ax3.grid(True, alpha=0.3)

# 4. 最大回撤对比
ax4 = axes[1, 1]
max_drawdowns = results_summary['max_drawdown'].sort_values(ascending=False)
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(max_drawdowns)))
bars = ax4.bar(max_drawdowns.index, max_drawdowns * 100, color=colors)
ax4.set_ylabel('最大回撤 (%)')
ax4.set_title('策略最大回撤对比', fontsize=12)
ax4.tick_params(axis='x', rotation=45)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../output/backtest_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. 结论

本Notebook复现了天风证券研报中的主成分风险平价模型，并进行了以下改进：

1. **PCRP模型**：利用PCA将高相关性资产变换到正交空间，实现真正的等风险贡献

2. **衰减加权法**：对波动率和相关系数使用指数衰减加权，更注重近期市场状态

3. **趋势跟踪法**：使用移动平均线交叉判断市场趋势，辅助预期收益估计

4. **WDC-PCRP**：结合衰减加权和趋势跟踪的完整模型

**注意**：本Notebook使用模拟数据进行演示。实际使用时需要：
- 使用有效的Tushare Pro token获取真实市场数据
- 调整资产配置和参数以适应不同市场环境
- 进行更严格的风控和仓位管理

In [ ]:
# 保存结果到CSV
results_summary_display.to_csv('../output/backtest_results_summary.csv')
print('结果已保存到 ../output/backtest_results_summary.csv')